# Downstream Classification Evaluation

Load trained LR / SR / HR classifiers from `crop_classification_experiments.py`, evaluate them on a held-out balanced subset, and compare performance to answer: **does super-resolution actually help classification?**

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import dataset_utils
from training_utils import (
    load_class_names,
    build_plant_dict,
    load_plant_balanced_image_file_sample,
)
from crop_classification_experiments import (
    infer_generator_config,
    load_generator,
    NUM_CLASSES,
)
from downstream_classifiers import SimpleCNNClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Discover Classifier Checkpoints

Scan `checkpoints/` for `classifier_*` subdirectories and group them by experiment (LR, SR, HR).

In [ ]:
CHECKPOINT_DIR = "checkpoints"

# Find all classifier checkpoint directories
classifier_dirs = sorted(
    d for d in Path(CHECKPOINT_DIR).iterdir()
    if d.is_dir() and d.name.startswith("classifier_")
)

# Group by experiment name: classifier_{lr,sr,hr}_{experiment} -> experiment
experiments = defaultdict(dict)
for d in classifier_dirs:
    name = d.name
    for variant in ("lr", "sr", "hr"):
        prefix = f"classifier_{variant}_"
        if name.startswith(prefix):
            experiment_name = name[len(prefix):]
            checkpoint_path = d / "checkpoint_latest.pt"
            if checkpoint_path.exists():
                experiments[experiment_name][variant] = checkpoint_path
            break

if not experiments:
    raise FileNotFoundError(
        "No classifier checkpoints found. Run crop_classification_experiments.py first."
    )

print("Available experiments:")
exp_list = sorted(experiments.keys())
for i, name in enumerate(exp_list):
    variants = sorted(experiments[name].keys())
    print(f"  [{i}] {name}  ({', '.join(variants)})")

In [ ]:
# --- Select which experiment to evaluate ---
# Option 1: Set EXPERIMENT_NAME directly (e.g. "agricultural_4blocks_total10epochs")
# Option 2: Set to None and use EXPERIMENT_IDX to pick from the list above
EXPERIMENT_NAME = None
EXPERIMENT_IDX = 0

if EXPERIMENT_NAME is not None:
    if EXPERIMENT_NAME not in experiments:
        raise KeyError(
            f"Experiment '{EXPERIMENT_NAME}' not found.\n"
            f"Available: {exp_list}"
        )
    experiment_name = EXPERIMENT_NAME
else:
    experiment_name = exp_list[EXPERIMENT_IDX]

exp_checkpoints = experiments[experiment_name]
print(f"Selected: {experiment_name}")
print(f"Variants: {sorted(exp_checkpoints.keys())}")

## 2. Load Classifiers

Instantiate a `SimpleCNNClassifier` for each variant (LR, SR, HR) and load the trained weights.

In [ ]:
classifiers = {}
for variant in ("lr", "sr", "hr"):
    if variant not in exp_checkpoints:
        print(f"  WARNING: no {variant.upper()} checkpoint found, skipping")
        continue
    model = SimpleCNNClassifier(num_classes=NUM_CLASSES).to(device)
    ckpt = torch.load(exp_checkpoints[variant], map_location=device, weights_only=False)
    model.load_state_dict(ckpt["classifier_state_dict"])
    model.eval()
    epoch = ckpt.get("epoch", "?")
    print(f"  {variant.upper()} classifier loaded (epoch {epoch})")
    classifiers[variant] = model

print(f"\nLoaded {len(classifiers)} classifier(s)")

## 3. Load SR Generator & Evaluation Data

Load the matching SR generator (needed for the SR classifier) and a balanced evaluation subset.

In [ ]:
# Load the SR generator that corresponds to this experiment
gen_checkpoint_path = Path(CHECKPOINT_DIR) / experiment_name / "checkpoint_latest.pt"
if not gen_checkpoint_path.exists():
    raise FileNotFoundError(f"Generator checkpoint not found: {gen_checkpoint_path}")

gen_type, num_blocks = infer_generator_config(gen_checkpoint_path)
generator = load_generator(str(gen_checkpoint_path), gen_type, num_blocks, device)
print(f"Generator: {gen_type}, blocks: {num_blocks}")

In [ ]:
DATA_DIR = "data/plantvillage/color_images/train"
SAMPLES_PER_CLASS = 50  # small balanced subset for evaluation

class_names = load_class_names(DATA_DIR)
plant_dict = build_plant_dict(class_names)
image_files, labels, used_class_names = load_plant_balanced_image_file_sample(
    DATA_DIR, plant_dict, SAMPLES_PER_CLASS
)

eval_dataset = dataset_utils.SuperResolutionDatasetWithLabels(
    image_files=image_files, image_labels=labels
)
eval_loader = DataLoader(eval_dataset, batch_size=16, shuffle=False, num_workers=2)
print(f"Evaluation set: {len(eval_dataset)} images, {len(class_names)} classes")

## 4. Evaluate All 3 Classifiers

Run each classifier on the evaluation set and collect predictions, accuracy, F1 scores, and confusion matrices.

In [ ]:
results = {}

for variant, model in classifiers.items():
    all_labels = []
    all_preds = []
    model.eval()

    with torch.no_grad():
        for batch_labels, low_res, high_res in eval_loader:
            batch_labels = batch_labels.to(device)
            low_res, high_res = low_res.to(device), high_res.to(device)

            if variant == "lr":
                outputs = model(low_res)
            elif variant == "sr":
                fake_hr = generator(low_res)
                outputs = model(fake_hr)
            else:  # hr
                outputs = model(high_res)

            _, predicted = torch.max(outputs, 1)
            all_labels.extend(batch_labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    per_class_f1 = f1_score(all_labels, all_preds, average=None)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)

    results[variant] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "per_class_f1": per_class_f1,
        "confusion_matrix": cm,
        "labels": all_labels,
        "preds": all_preds,
    }

    print(f"\n{'='*60}")
    print(f"  {variant.upper()} Classifier")
    print(f"{'='*60}")
    print(report)
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}")

## 5. Head-to-Head Comparison

Grouped bar chart comparing overall accuracy and macro F1 for LR, SR, and HR classifiers.

In [ ]:
variants = [v for v in ("lr", "sr", "hr") if v in results]
labels_display = {"lr": "LR (64×64)", "sr": "SR (Generated)", "hr": "HR (256×256)"}
colors = {"lr": "#e74c3c", "sr": "#3498db", "hr": "#2ecc71"}

metrics_names = ["Accuracy", "Macro F1"]
x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, v in enumerate(variants):
    values = [results[v]["accuracy"], results[v]["macro_f1"]]
    bars = ax.bar(x + i * width, values, width, label=labels_display[v], color=colors[v])
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10,
        )

ax.set_ylabel("Score")
ax.set_title(f"Classifier Comparison — {experiment_name}")
ax.set_xticks(x + width * (len(variants) - 1) / 2)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("eval_classifier_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_classifier_comparison.png")

## 6. Per-Class F1 Breakdown

Horizontal bar chart showing F1 per disease class for all 3 variants. Highlights which classes benefit most from super-resolution.

In [ ]:
# Only include classes that appear in the evaluation set
unique_labels = sorted(set(results[variants[0]]["labels"]))
display_names = [class_names[i] if i < len(class_names) else f"class_{i}" for i in unique_labels]
n_classes = len(unique_labels)

fig, ax = plt.subplots(figsize=(10, max(6, n_classes * 0.35)))
y = np.arange(n_classes)
bar_height = 0.25

for i, v in enumerate(variants):
    f1_vals = results[v]["per_class_f1"]
    # per_class_f1 is indexed by label position in sklearn's output, align to unique_labels
    ax.barh(
        y + i * bar_height, f1_vals[:n_classes], bar_height,
        label=labels_display[v], color=colors[v], alpha=0.85,
    )

ax.set_xlabel("F1 Score")
ax.set_title(f"Per-Class F1 — {experiment_name}")
ax.set_yticks(y + bar_height * (len(variants) - 1) / 2)
ax.set_yticklabels(display_names, fontsize=8)
ax.set_xlim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("eval_classifier_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_classifier_per_class_f1.png")

## 7. Confusion Matrices

Side-by-side confusion matrices for LR, SR, and HR classifiers.

In [ ]:
n_plots = len(variants)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

for ax, v in zip(axes, variants):
    cm = results[v]["confusion_matrix"]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"{labels_display[v]}\nAcc={results[v]['accuracy']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(f"Confusion Matrices — {experiment_name}", fontsize=14)
plt.tight_layout()
plt.savefig("eval_classifier_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_classifier_confusion_matrices.png")

## 8. Training Curves & Summary

Load sequential epoch checkpoints for each classifier and plot training loss curves. Print a final summary table.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for v in variants:
    model_dir = Path(CHECKPOINT_DIR) / f"classifier_{v}_{experiment_name}"
    epoch_files = sorted(model_dir.glob("checkpoint_epoch_*.pt"))
    if not epoch_files:
        print(f"  No epoch checkpoints for {v.upper()}, skipping curve")
        continue

    epochs = []
    losses = []
    for ef in epoch_files:
        ckpt = torch.load(ef, map_location="cpu", weights_only=False)
        epochs.append(ckpt["epoch"])
        losses.append(ckpt["running_loss"])

    ax.plot(epochs, losses, marker="o", markersize=4, label=labels_display[v], color=colors[v])

ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title(f"Classifier Training Curves — {experiment_name}")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("eval_classifier_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_classifier_training_curves.png")

In [ ]:
print("\n" + "=" * 50)
print("  SUMMARY")
print("=" * 50)
print(f"Experiment: {experiment_name}")
print(f"Eval set:   {len(eval_dataset)} images, {n_classes} classes")
print()
print(f"{'Variant':<20} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 42)
for v in variants:
    label = labels_display[v]
    acc = results[v]["accuracy"]
    f1 = results[v]["macro_f1"]
    print(f"{label:<20} {acc:>10.4f} {f1:>10.4f}")

# Highlight SR vs LR improvement
if "lr" in results and "sr" in results:
    lr_acc = results["lr"]["accuracy"]
    sr_acc = results["sr"]["accuracy"]
    delta = sr_acc - lr_acc
    direction = "+" if delta >= 0 else ""
    print(f"\nSR vs LR accuracy delta: {direction}{delta:.4f}")
    lr_f1 = results["lr"]["macro_f1"]
    sr_f1 = results["sr"]["macro_f1"]
    delta_f1 = sr_f1 - lr_f1
    direction_f1 = "+" if delta_f1 >= 0 else ""
    print(f"SR vs LR macro F1 delta: {direction_f1}{delta_f1:.4f}")

print("\nFigures saved:")
print("  eval_classifier_comparison.png")
print("  eval_classifier_per_class_f1.png")
print("  eval_classifier_confusion_matrices.png")
print("  eval_classifier_training_curves.png")